In [ ]:
# Ajustar el directorio de trabajo al de la raíz del proyecto si ejecutamos desde la carpeta notebooks
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print("Directorio de trabajo actual:", os.getcwd())


In [1]:
'''pip install h3
pip install kaleido
pip install plotly
pip install seaborn
pip install matplotlib
pip install duckdb'''

'pip install h3\npip install kaleido\npip install plotly\npip install seaborn\npip install matplotlib\npip install duckdb'

In [2]:
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
import plotly.express as px
import kaleido
import h3

In [3]:
# Configuración de estilo para los gráficos
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'font.size': 12, 'axes.titlesize': 14})

In [4]:
def run_eda():
    print("Iniciando el Análisis Exploratorio de Datos (EDA) con DuckDB...")

    # Construir el contorno de tierra de AMBA desde indices_h3.csv
    import pandas as pd
    from shapely.geometry import Polygon, Point
    from shapely.ops import unary_union
    from shapely.prepared import prep
    
    df_indices = pd.read_csv("data/indices_h3.csv")
    df_indices['h3_res8'] = df_indices['h3'].apply(lambda x: h3.cell_to_parent(x, 8))
    unique_res8 = df_indices['h3_res8'].unique()
    polygons = []
    for hex_id in unique_res8:
        boundary_lat_lon = h3.cell_to_boundary(hex_id)
        polygons.append(Polygon([(lon, lat) for lat, lon in boundary_lat_lon]))
    land_boundary = unary_union(polygons)
    prepared_land = prep(land_boundary)
    
    # Conexión principal de DuckDB en memoria
    con = duckdb.connect(database=':memory:')
    
    # 1. CARGA DE DATOS COMO VISTAS DE DUCKDB
    # ========================================
    try:
        print("Cargando datasets en DuckDB...")
        # Configurar DuckDB para que lea los CSV de manera óptima
        con.execute("CREATE VIEW viajes AS SELECT * FROM read_csv_auto('resultados/viajes.csv', ignore_errors=true)")
        con.execute("CREATE VIEW etapas AS SELECT * FROM read_csv_auto('resultados/etapas.csv', ignore_errors=true)")
        con.execute("CREATE VIEW tarjetas AS SELECT * FROM read_csv_auto('resultados/tarjetas.csv', ignore_errors=true)")
        con.execute("CREATE VIEW indices_h3 AS SELECT * FROM read_csv_auto('data/indices_h3.csv', ignore_errors=true)")
        con.execute("CREATE VIEW paradas AS SELECT * FROM read_csv_auto('data/paradas.csv', ignore_errors=true)")
        con.execute("CREATE VIEW tarifas AS SELECT * FROM read_csv_auto('data/tarifas.csv', ignore_errors=true)")
    except Exception as e:
        print(f"Error al cargar los archivos: {e}")
        print("Asegurate de descargar los .csv del repositorio y mantener la estructura de carpetas.")
        return

    # 2. CALIDAD Y LIMPIEZA DE DATOS
    # ==============================
    print("\n--- Estadísticas Generales ---")
    stats = con.execute("""
        SELECT 
            (SELECT COUNT(*) FROM etapas) AS total_etapas,
            (SELECT COUNT(*) FROM etapas WHERE parada_id_d IS NOT NULL) AS etapas_validas,
            (SELECT COUNT(DISTINCT id_tarjeta) FROM tarjetas) AS total_tarjetas,
            (SELECT COUNT(*) FROM viajes) AS total_viajes
    """).fetchone()
    
    total_etapas, etapas_validas, total_tarjetas, total_viajes = stats
    print(f"Total de etapas válidas: {total_etapas:,}")
    print(f"Total de tarjetas únicas: {total_tarjetas:,}")
    print(f"Total de viajes completos: {total_viajes:,}")
    
    if total_etapas > 0:
        tasa_exito = (etapas_validas / total_etapas) * 100
        print(f"Tasa de éxito en imputación de destinos (etapas): {tasa_exito:.1f}%")

    # 3. CRUCE ESPACIAL, CÁLCULO DE DISTANCIAS Y CLASIFICACIÓN
    # ========================================================
    print("\nProcesando tipologías geográficas, distancias y tarifas con SQL...")
    
    # Creamos una vista maestra `viajes_geo` que resuelve todo: coordenadas, haversine, modos, y tarifas.
    con.execute("""
        CREATE VIEW viajes_geo AS
        WITH viajes_coords AS (
            SELECT 
                v.*,
                po.latitud AS lat_o, po.longitud AS lon_o, po.h3 AS h3_o,
                pd.latitud AS lat_d, pd.longitud AS lon_d, pd.h3 AS h3_d,
                
                -- Regex para limpiar las terminaciones .0 en id_tarjeta e id_viaje
                REGEXP_REPLACE(CAST(v.id_tarjeta AS VARCHAR), '\\.0$', '') AS id_tarjeta_str,
                REGEXP_REPLACE(CAST(v.id_viaje AS VARCHAR), '\\.0$', '') AS id_viaje_str,

                -- Distancia Haversine pura en SQL
                6371.0 * 2 * ASIN(SQRT(
                    POWER(SIN(RADIANS(pd.latitud - po.latitud) / 2.0), 2) + 
                    COS(RADIANS(po.latitud)) * COS(RADIANS(pd.latitud)) * POWER(SIN(RADIANS(pd.longitud - po.longitud) / 2.0), 2)
                )) AS distancia,
                
                -- Jurisdicciones
                CASE WHEN h3_o_idx.provincia ILIKE '%CABA%' OR h3_o_idx.provincia ILIKE '%Ciudad Autónoma%' OR h3_o_idx.provincia ILIKE '%Capital Federal%' THEN 'CABA' ELSE 'PBA' END AS jur_origen,
                CASE WHEN h3_d_idx.provincia ILIKE '%CABA%' OR h3_d_idx.provincia ILIKE '%Ciudad Autónoma%' OR h3_d_idx.provincia ILIKE '%Capital Federal%' THEN 'CABA' ELSE 'PBA' END AS jur_destino
                
            FROM viajes v
            LEFT JOIN paradas po ON v.parada_id_o = po.id
            LEFT JOIN paradas pd ON v.parada_id_d = pd.id
            LEFT JOIN indices_h3 h3_o_idx ON po.h3 = h3_o_idx.h3
            LEFT JOIN indices_h3 h3_d_idx ON pd.h3 = h3_d_idx.h3
        ),
        viajes_clasificados AS (
            SELECT 
                *,
                CASE 
                    WHEN jur_origen = 'CABA' AND jur_destino = 'CABA' THEN 'Intra-CABA'
                    WHEN jur_origen = 'PBA' AND jur_destino = 'PBA' THEN 'Intra-PBA'
                    WHEN jur_origen IS NOT NULL AND jur_destino IS NOT NULL THEN 'CABA-PBA'
                    ELSE 'Desconocido'
                END AS tipo_viaje,
                CASE 
                    WHEN jur_origen = 'CABA' AND jur_destino = 'CABA' THEN 'Intra-CABA'
                    WHEN jur_origen = 'PBA' AND jur_destino = 'PBA' THEN 'Intra-PBA'
                    WHEN jur_origen = 'PBA' AND jur_destino = 'CABA' THEN 'PBA-CABA'
                    WHEN jur_origen = 'CABA' AND jur_destino = 'PBA' THEN 'CABA-PBA'
                    ELSE 'Desconocido'
                END AS tipo_viaje_detallado,
                CASE 
                    WHEN jur_origen = 'CABA' AND jur_destino = 'CABA' THEN 'Intra-CABA'
                    WHEN jur_origen = 'PBA' AND jur_destino = 'PBA' THEN 'Intra-PBA'
                    WHEN jur_origen = 'PBA' AND jur_destino = 'CABA' THEN 'PBA-CABA'
                    WHEN jur_origen = 'CABA' AND jur_destino = 'PBA' THEN 'CABA-PBA'
                    ELSE 'Desconocido'
                END AS tipo_viaje_detallado,
                
                CASE 
                    WHEN cantidad_etapas = 1 THEN '1 etapa'
                    WHEN cantidad_etapas = 2 THEN '2 etapas'
                    ELSE '3+ etapas'
                END AS etapas_agrupadas,
                
                -- Modos de Transporte
                COALESCE(etapas_colectivo, 0) AS c_col,
                COALESCE(etapas_tren, 0) AS c_tren,
                COALESCE(etapas_subte, 0) AS c_sub
            FROM viajes_coords
        ),
        viajes_modos AS (
            SELECT
                *,
                CASE
                    WHEN c_col > 0 AND c_tren = 0 AND c_sub = 0 THEN 'Solo Colectivo'
                    WHEN c_tren > 0 AND c_col = 0 AND c_sub = 0 THEN 'Solo Tren'
                    WHEN c_sub > 0 AND c_col = 0 AND c_tren = 0 THEN 'Solo Subte'
                    WHEN c_col > 0 AND c_tren > 0 AND c_sub = 0 THEN 'Col + Tren'
                    WHEN c_col > 0 AND c_sub > 0 AND c_tren = 0 THEN 'Col + Subte'
                    WHEN c_tren > 0 AND c_sub > 0 AND c_col = 0 THEN 'Tren + Subte'
                    WHEN c_col > 0 AND c_tren > 0 AND c_sub > 0 THEN 'Col + Tren + Subte'
                    ELSE 'Desconocido/Otros'
                END AS modo_transporte
            FROM viajes_clasificados
            WHERE tipo_viaje != 'Desconocido'
        ),
        -- Extraer primera tarifa válida por tarjeta y viaje desde etapas
        tarifas_unicas AS (
            SELECT 
                REGEXP_REPLACE(CAST(id_tarjeta AS VARCHAR), '\\.0$', '') AS id_tarjeta_str,
                REGEXP_REPLACE(CAST(id_viaje AS VARCHAR), '\\.0$', '') AS id_viaje_str,
                FIRST(CAST(id_tarifa AS INTEGER)) AS id_tarifa,
                MIN(TRY_CAST(hora AS INTEGER)) AS hora_inicio,
                MAX(TRY_CAST(hora AS INTEGER)) AS hora_fin
            FROM etapas
            WHERE id_tarifa IS NOT NULL
            GROUP BY 1, 2
        )
        SELECT 
            v.*,
            COALESCE(t.nombre, 'Desconocida') AS nombre_tarifa,
            CASE 
                WHEN UPPER(COALESCE(t.nombre, 'Desconocida')) IN ('TARIFA PLENA', 'DESCONOCIDA') THEN 'Tarifa Plena'
                ELSE 'Tarifa Social / Subsidio'
            END AS perfil_tarifario,
            
            -- Tiempos y Fricción
            tu.hora_inicio,
            tu.hora_fin,
            CASE 
                WHEN (tu.hora_fin - tu.hora_inicio) < 0 THEN (tu.hora_fin - tu.hora_inicio) + 24 
                WHEN (tu.hora_fin - tu.hora_inicio) = 0 THEN 
                    LEAST(
                        CASE 
                            WHEN modo_transporte = 'Solo Colectivo' THEN (distancia / 15.0) + 0.08
                            WHEN modo_transporte = 'Solo Tren' THEN (distancia / 35.0) + 0.15
                            WHEN modo_transporte = 'Solo Subte' THEN (distancia / 25.0) + 0.08
                            WHEN modo_transporte = 'Col + Tren' THEN (distancia / 25.0) + 0.25
                            WHEN modo_transporte = 'Col + Subte' THEN (distancia / 20.0) + 0.16
                            WHEN modo_transporte = 'Tren + Subte' THEN (distancia / 30.0) + 0.23
                            WHEN modo_transporte = 'Col + Tren + Subte' THEN (distancia / 25.0) + 0.33
                            ELSE (distancia / 18.0) + 0.1
                        END, 
                        0.9
                    )
                ELSE COALESCE((tu.hora_fin - tu.hora_inicio), 0)
            END AS duracion_horas
        FROM viajes_modos v
        LEFT JOIN tarifas_unicas tu ON v.id_tarjeta_str = tu.id_tarjeta_str AND v.id_viaje_str = tu.id_viaje_str
        LEFT JOIN tarifas t ON tu.id_tarifa = CAST(t.id AS INTEGER)
    """)

    # 4. FIGURA 1: Intermodalidad por Geografía
    # =========================================
    print("\nGenerando Figura 1...")
    prop_df = con.execute("""
        SELECT 
            tipo_viaje, 
            etapas_agrupadas, 
            COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(PARTITION BY tipo_viaje) AS porcentaje 
        FROM viajes_geo 
        GROUP BY tipo_viaje, etapas_agrupadas
    """).df()

    print("\n--- DATOS FIGURA 1 (Porcentaje de viajes por etapas) ---")
    tabla_fig1 = prop_df.pivot(index='tipo_viaje', columns='etapas_agrupadas', values='porcentaje').round(1)
    print(tabla_fig1)

    plt.figure(figsize=(10, 6))
    ax1 = sns.barplot(data=prop_df, x='tipo_viaje', y='porcentaje', hue='etapas_agrupadas',
                      order=['Intra-CABA', 'Intra-PBA', 'CABA-PBA'],
                      hue_order=['1 etapa', '2 etapas', '3+ etapas'])
    for container in ax1.containers:
        ax1.bar_label(container, fmt='%.1f%%', padding=3, size=10)
    plt.title('Figura 1: Porcentaje de viajes según cantidad de etapas por zona', fontweight='bold')
    plt.xlabel('Tipo de Viaje')
    plt.ylabel('Porcentaje (%)')
    plt.legend(title='Cantidad de Etapas')
    plt.tight_layout()
    plt.savefig('resultados/visualizaciones/figura1_intermodalidad.png', dpi=300)
    plt.close()

    # 5. FIGURA 2: Distancias y Outliers por Geografía
    # ================================================
    print("\nGenerando Figura 2...")
    viajes_plot = con.execute("""
        SELECT tipo_viaje, distancia 
        FROM viajes_geo 
        WHERE distancia > 0 AND distancia < 100
    """).df()

    print("\n--- DATOS FIGURA 2 (Estadísticas de distancia en km) ---")
    stats_dist = viajes_plot.groupby('tipo_viaje')['distancia'].describe()[['count', 'mean', '50%', 'max']].round(1)
    stats_dist.rename(columns={'count': 'Cantidad', 'mean': 'Promedio', '50%': 'Mediana', 'max': 'Máximo'}, inplace=True)
    print(stats_dist)

    plt.figure(figsize=(10, 6))
    ax2 = sns.boxplot(data=viajes_plot, x='tipo_viaje', y='distancia',
                      order=['Intra-CABA', 'Intra-PBA', 'CABA-PBA'], hue='tipo_viaje', legend=False,
                      palette='Set2', showfliers=True, flierprops={'marker': 'o', 'markersize': 3, 'alpha': 0.3})
    
    medians = viajes_plot.groupby('tipo_viaje')['distancia'].median()
    orden = ['Intra-CABA', 'Intra-PBA', 'CABA-PBA']
    for i, categoria in enumerate(orden):
        if categoria in medians:
            valor_mediana = medians[categoria]
            ax2.text(i, valor_mediana + 0.5, f'{valor_mediana:.1f} km', weight='bold',
                     horizontalalignment='center', size=11, color='black',
                     bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=1))
    
    plt.title('Figura 2: Distribución de distancias de viaje según zona geográfica', fontweight='bold')
    plt.xlabel('Tipo de Viaje')
    plt.ylabel('Distancia (km)')
    plt.tight_layout()
    plt.savefig('resultados/visualizaciones/figura2_distancias.png', dpi=300)
    plt.close()

    # 6. FIGURA 3: Modo de Transporte
    # =========================================
    print("\nGenerando Figura 3...")
    prop_modo = con.execute("""
        SELECT 
            tipo_viaje, 
            modo_transporte, 
            COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(PARTITION BY tipo_viaje) AS porcentaje 
        FROM viajes_geo 
        GROUP BY tipo_viaje, modo_transporte
    """).df()

    print("\n--- DATOS FIGURA 3 (Modos de transporte % por zona) ---")
    tabla_fig3 = prop_modo.pivot(index='tipo_viaje', columns='modo_transporte', values='porcentaje').fillna(0).round(1)
    print(tabla_fig3)

    plt.figure(figsize=(12, 7))
    orden_modos = ['Solo Colectivo', 'Col + Tren', 'Solo Tren', 'Col + Subte', 'Solo Subte', 'Col + Tren + Subte', 'Tren + Subte']
    orden_modos_presentes = [m for m in orden_modos if m in prop_modo['modo_transporte'].unique()]
    
    ax3 = sns.barplot(data=prop_modo, x='tipo_viaje', y='porcentaje', hue='modo_transporte',
                      order=['Intra-CABA', 'Intra-PBA', 'CABA-PBA'], hue_order=orden_modos_presentes, palette='tab10')
    
    for container in ax3.containers:
        labels = [f'{v.get_height():.1f}%' if v.get_height() > 0.5 else '' for v in container]
        ax3.bar_label(container, labels=labels, padding=3, size=9)

    plt.title('Figura 3: Composición Modal de Viajes por Jurisdicción', fontweight='bold')
    plt.xlabel('Tipo de Viaje')
    plt.ylabel('Porcentaje (%)')
    plt.legend(title='Modo/Combinación', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig('resultados/visualizaciones/figura3_modos.png', dpi=300)
    plt.close()

    # 7. FIGURA 4: Perfil Tarifario por Zona
    # ======================================
    print("\nGenerando Figura 4 (Tarifas agrupadas)...")
    prop_tarifa = con.execute("""
        SELECT 
            tipo_viaje, 
            perfil_tarifario, 
            COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(PARTITION BY tipo_viaje) AS porcentaje 
        FROM viajes_geo 
        WHERE nombre_tarifa != 'Desconocida'
        GROUP BY tipo_viaje, perfil_tarifario
    """).df()

    print("\n--- DATOS FIGURA 4 (Perfil Tarifario % por zona) ---")
    tabla_fig4 = prop_tarifa.pivot(index='tipo_viaje', columns='perfil_tarifario', values='porcentaje').fillna(0).round(1)
    print(tabla_fig4)

    plt.figure(figsize=(10, 6))
    ax4 = sns.barplot(data=prop_tarifa, x='tipo_viaje', y='porcentaje', hue='perfil_tarifario',
                      order=['Intra-CABA', 'Intra-PBA', 'CABA-PBA'], palette='Set1')
    
    for container in ax4.containers:
        ax4.bar_label(container, fmt='%.1f%%', padding=3, size=11, weight='bold')

    plt.title('Figura 4: Distribución de Tarifa Social vs Plena por Jurisdicción', fontweight='bold')
    plt.xlabel('Tipo de Viaje')
    plt.ylabel('Porcentaje (%)')
    plt.legend(title='Perfil Tarifario')
    plt.tight_layout()
    plt.savefig('resultados/visualizaciones/figura4_tarifas.png', dpi=300)
    plt.close()

    # 8. FIGURA 5: Desglose Específico de Viajes y Etapas por Tarifa
    # ==============================================================
    print("\nGenerando Figura 5 (Desglose exacto de tarifas)...")
    resumen_tarifas = con.execute("""
        WITH viajes_t AS (
            SELECT nombre_tarifa, COUNT(*) AS Viajes 
            FROM viajes_geo 
            WHERE nombre_tarifa != 'Desconocida' 
            GROUP BY nombre_tarifa
        ),
        etapas_t AS (
            SELECT COALESCE(t.nombre, 'Desconocida') AS nombre_tarifa, COUNT(*) AS Etapas_Totales
            FROM etapas e
            LEFT JOIN tarifas t ON CAST(e.id_tarifa AS INTEGER) = CAST(t.id AS INTEGER)
            WHERE e.id_tarifa IS NOT NULL AND COALESCE(t.nombre, 'Desconocida') != 'Desconocida'
            GROUP BY t.nombre
        )
        SELECT 
            v.nombre_tarifa, 
            COALESCE(v.Viajes, 0) AS Viajes, 
            COALESCE(e.Etapas_Totales, 0) AS Etapas_Totales,
            CASE WHEN v.Viajes > 0 THEN ROUND(e.Etapas_Totales * 1.0 / v.Viajes, 2) ELSE 0.0 END AS Prom_Etapas_por_Viaje
        FROM viajes_t v
        FULL OUTER JOIN etapas_t e ON v.nombre_tarifa = e.nombre_tarifa
        ORDER BY Etapas_Totales DESC
    """).df()

    print("\n--- DATOS FIGURA 5 (Viajes y Etapas por Tarifa Específica) ---")
    print(resumen_tarifas.to_string(index=False, formatters={'Viajes': '{:,.0f}'.format, 'Etapas_Totales': '{:,.0f}'.format}))

    plt.figure(figsize=(12, 8))
    top_tarifas = resumen_tarifas.head(15)
    ax5 = sns.barplot(data=top_tarifas, x='Etapas_Totales', y='nombre_tarifa', hue='nombre_tarifa', palette='viridis', legend=False)
    
    for container in ax5.containers:
        ax5.bar_label(container, fmt='{:,.0f}', padding=5, size=10)
    
    ax5.get_xaxis().set_major_formatter(plt.FuncFormatter(lambda x, loc: "{:,}".format(int(x))))
    plt.title('Figura 5: Volumen de Etapas según Tipo de Tarifa Específica', fontweight='bold')
    plt.xlabel('Cantidad de Etapas Totales')
    plt.ylabel('Tipo de Tarifa')
    plt.tight_layout()
    plt.savefig('resultados/visualizaciones/figura5_detalle_tarifas.png', dpi=300)
    plt.close()

    # 9. FIGURA 6: Mapa de Calor Optimizado (Spatial Binning)
    # =======================================================
    print("\nGenerando Figura 6 (Mapa de Densidad Optimizado)...")
    try:
        con.execute("CREATE OR REPLACE VIEW transacciones AS SELECT * FROM read_csv_auto('data/transacciones.csv', ignore_errors=true)")
        
        # EL TRUCO: Redondear a 3 decimales crea una grilla espacial de ~110x110 metros.
        # Agrupamos por esa grilla y contamos las transacciones usando el motor de base de datos.
        mapa_agrupado = con.execute("""
            SELECT 
                ROUND(TRY_CAST(lon AS DOUBLE), 4) AS lon_bin, 
                ROUND(TRY_CAST(lat AS DOUBLE), 4) AS lat_bin, 
                COUNT(*) AS peso_densidad
            FROM transacciones 
            WHERE lon IS NOT NULL AND lat IS NOT NULL
            GROUP BY 1, 2
        """).df()
        if not mapa_agrupado.empty:
            # Filtrar puntos fuera del contorno de tierra de AMBA
            mapa_agrupado['in_land'] = mapa_agrupado.apply(lambda r: prepared_land.contains(Point(r['lon_bin'], r['lat_bin'])), axis=1)
            mapa_agrupado = mapa_agrupado[mapa_agrupado['in_land']].drop(columns=['in_land'])
            # Ahora le pasamos a Plotly el 'peso_densidad' (z) para que sepa qué tan "caliente" es cada punto
            fig = px.density_map(
                mapa_agrupado, 
                lat='lat_bin', 
                lon='lon_bin', 
                z='peso_densidad',      # Variable de intensidad
                radius=32,               # Tamaño de la mancha de calor
                center=dict(lat=-34.6037, lon=-58.3816), # Centro en el AMBA (Obelisco)
                zoom=10,
                map_style="carto-positron", 
                title="Figura 6: Densidad Espacial de Transacciones en el AMBA (Optimizado)"
            )
            
            fig.write_html('resultados/visualizaciones/figura6_mapa_transacciones_optimizado.html')
            print(f"Figura 6 guardada con éxito. Tamaño drásticamente reducido (Puntos ploteados: {len(mapa_agrupado):,}).")
            
        else:
            print("Aviso: No se encontraron datos numéricos válidos en las columnas 'lat' y 'lon'.")
            
    except Exception as e:
        print(f"Error al generar el mapa optimizado: {e}")

    # 10. FIGURA 7: Mapa Hexagonal H3 (AMBA)
    # ======================================
    print("\nGenerando Figura 7 (Mapa H3)...")
    try:
        # 1. Instalar y cargar la extensión H3 nativa de DuckDB
        con.execute("INSTALL h3 FROM community;")
        con.execute("LOAD h3;")
        
        # (Opcional si ya se creó en la Figura 6, pero lo dejamos por seguridad)
        con.execute("CREATE OR REPLACE VIEW transacciones AS SELECT * FROM read_csv_auto('data/transacciones.csv', ignore_errors=true)")
        
        # 2. Agrupación espacial usando H3 en resolución 8 (aprox 730m)
        print("Agregando datos espacialmente con DuckDB H3...")
        mapa_h3 = con.execute("""
            SELECT 
                h3_latlng_to_cell_string(TRY_CAST(lat AS DOUBLE), TRY_CAST(lon AS DOUBLE), 8) AS h3_index,
                COUNT(*) AS cantidad_transacciones
            FROM transacciones 
            WHERE lat IS NOT NULL AND lon IS NOT NULL
            GROUP BY 1
        """).df()
        if not mapa_h3.empty:
            # Filtrar hexágonos fuera del contorno de tierra de AMBA
            mapa_h3['centroid'] = mapa_h3['h3_index'].apply(lambda x: h3.cell_to_latlng(x))
            mapa_h3['in_land'] = mapa_h3.apply(lambda r: prepared_land.contains(Point(r['centroid'][1], r['centroid'][0])), axis=1)
            mapa_h3 = mapa_h3[mapa_h3['in_land']].drop(columns=['centroid', 'in_land'])
            print("Construyendo geometrías poligonales para Plotly...")
            
            # 3. Construir el GeoJSON necesario para Plotly (Actualizado para H3 v4)
            geojson_features = []
            for hex_id in mapa_h3['h3_index']:
                # Obtener vértices en formato (lat, lon)
                boundary_lat_lon = h3.cell_to_boundary(hex_id)
                
                # Invertir a [lon, lat] para cumplir con el estándar GeoJSON
                boundary = [[lon, lat] for lat, lon in boundary_lat_lon]
                
                # Cerramos el polígono repitiendo el primer punto al final
                boundary.append(boundary[0])
                
                geojson_features.append({
                    "type": "Feature",
                    "id": hex_id,
                    "geometry": {
                        "type": "Polygon",
                        "coordinates": [boundary] # GeoJSON requiere una lista de listas de coordenadas
                    }
                })
                
            geojson_data = {
                "type": "FeatureCollection",
                "features": geojson_features
            }

           # 4. Generar el mapa Choropleth interactivo
            print("Renderizando mapa Choropleth (Figura 7)...")
            
            # EL TRUCO: Calculamos el percentil 95 para saturar el color máximo.
            # (Si todavía se ve muy oscuro, bajalo a 0.90. Si brilla demasiado, subilo a 0.98 o 0.99)
            umbral_maximo = mapa_h3['cantidad_transacciones'].quantile(0.98)
            
            fig7 = px.choropleth_map(
                mapa_h3,
                geojson=geojson_data,
                locations='h3_index',
                color='cantidad_transacciones',
                color_continuous_scale="Viridis",
                range_color=(0, umbral_maximo), # <--- ESTO MAGNIFICA EL RESTO DE LOS HEXÁGONOS
                map_style="carto-darkmatter",
                zoom=10,
                center=dict(lat=-34.6037, lon=-58.3816),
                opacity=0.7,
                title="Figura 7: Densidad de Transacciones en el AMBA (Índices H3 - Res 8)"
            )
            
            # Ajustar márgenes
            fig7.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
            
            # Guardar resultado en HTML para mantener interactividad
            fig7.write_html('resultados/visualizaciones/figura7_mapa_h3.html')
            print(f"Figura 7 guardada con éxito (figura7_mapa_h3.html). Total de hexágonos: {len(mapa_h3):,}")
            
        else:
            print("Aviso: No se encontraron datos numéricos válidos en las columnas 'lat' y 'lon' para la Figura 7.")
            
    except Exception as e:
        print(f"Error al generar la Figura 7 (Mapa H3): {e}")

    # 11. FIGURA 8: Mapa Hexagonal H3 Animado por Hora
    # ==================================================
    print("\nGenerando Figura 8 (Mapa H3 Animado por Hora)...")
    try:
        # Asumimos que tienes una columna 'hora' (entero de 0 a 23).
        # Si tienes una fecha/hora completa, usa: EXTRACT(HOUR FROM TRY_CAST(tu_columna AS TIMESTAMP)) AS hora_del_dia
        print("Agregando datos espacial y temporalmente...")
        mapa_h3_temporal = con.execute("""
            SELECT 
                h3_latlng_to_cell_string(TRY_CAST(lat AS DOUBLE), TRY_CAST(lon AS DOUBLE), 8) AS h3_index,
                TRY_CAST(hora AS INTEGER) AS hora_del_dia,
                COUNT(*) AS cantidad_transacciones
            FROM transacciones 
            WHERE lat IS NOT NULL AND lon IS NOT NULL AND hora IS NOT NULL
            GROUP BY 1, 2
            ORDER BY hora_del_dia ASC
        """).df()

        if not mapa_h3_temporal.empty:
            # Filtrar hexágonos fuera del contorno de tierra de AMBA
            mapa_h3_temporal['centroid'] = mapa_h3_temporal['h3_index'].apply(lambda x: h3.cell_to_latlng(x))
            mapa_h3_temporal['in_land'] = mapa_h3_temporal.apply(lambda r: prepared_land.contains(Point(r['centroid'][1], r['centroid'][0])), axis=1)
            mapa_h3_temporal = mapa_h3_temporal[mapa_h3_temporal['in_land']].drop(columns=['centroid', 'in_land'])
            print("Construyendo geometrías poligonales globales...")
            
            # 1. Extraemos los hexágonos ÚNICOS de todas las horas para armar el mapa base
            hex_unicos = mapa_h3_temporal['h3_index'].unique()
            
            geojson_features = []
            for hex_id in hex_unicos:
                boundary_lat_lon = h3.cell_to_boundary(hex_id)
                boundary = [[lon, lat] for lat, lon in boundary_lat_lon]
                boundary.append(boundary[0]) # Cerramos el polígono
                
                geojson_features.append({
                    "type": "Feature",
                    "id": hex_id,
                    "geometry": {
                        "type": "Polygon",
                        "coordinates": [boundary]
                    }
                })
                
            geojson_data = {
                "type": "FeatureCollection",
                "features": geojson_features
            }

            # 2. Calculamos el umbral máximo GLOBAL para congelar la paleta de colores
            umbral_maximo_global = mapa_h3_temporal['cantidad_transacciones'].quantile(0.98)

            # 3. Generar la animación
            print("Renderizando mapa Choropleth Animado (Figura 8)...")
            fig8 = px.choropleth_map(
                mapa_h3_temporal,
                geojson=geojson_data,
                locations='h3_index',
                color='cantidad_transacciones',
                animation_frame='hora_del_dia',  # <--- LA MAGIA ESTÁ ACÁ
                color_continuous_scale="Inferno", # Cambié a Inferno porque suele verse mejor en animaciones oscuras
                range_color=(0, umbral_maximo_global), # <--- Escala de color fija para todos los frames
                map_style="carto-darkmatter",
                zoom=10,
                center=dict(lat=-34.6037, lon=-58.3816),
                opacity=0.7,
                title="Figura 8: Pulso de Transacciones en el AMBA por Hora"
            )
            
            fig8.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
            
            # Guardamos el HTML (este archivo pesará un poco más que el anterior, dale unos segundos para renderizar)
            fig8.write_html('resultados/visualizaciones/figura8_mapa_h3_animado.html')
            print("Figura 8 guardada con éxito. ¡Abre el HTML para darle Play a la animación!")
            
        else:
            print("Aviso: No se generaron datos. Verifica que la columna de tiempo se llame 'hora'.")
            
    except Exception as e:
        print(f"Error al generar la Figura 8 (Animación): {e}")

    # 12. FIGURA 9: Análisis OD Animado del Hexágono (Cálculo H3 Dinámico)
    # ====================================================================
    print("\nGenerando Figura 9 (Flujos OD para el Hexágono 88c2e3022bfffff)...")
    try:
        hex_objetivo = '88c2e3022bfffff'
        
        lat_centro, lon_centro = h3.cell_to_latlng(hex_objetivo)
        
        print("Calculando índices H3 al vuelo (Resolución 8) para asegurar compatibilidad...")
        
        
        od_hex = con.execute(f"""
            WITH etapas_hora AS (
                SELECT 
                    TRY_CAST(TRY_CAST(id_tarjeta AS DOUBLE) AS BIGINT) AS id_tarjeta_num,
                    TRY_CAST(TRY_CAST(id_viaje AS DOUBLE) AS BIGINT) AS id_viaje_num,
                    MIN(TRY_CAST(hora AS INTEGER)) AS hora_viaje
                FROM read_csv_auto('resultados/etapas.csv', ignore_errors=true)
                WHERE id_tarjeta IS NOT NULL AND id_viaje IS NOT NULL
                GROUP BY 1, 2
            ),
            viajes_base AS (
                SELECT 
                    TRY_CAST(TRY_CAST(v.id_tarjeta AS DOUBLE) AS BIGINT) AS id_tarjeta_num,
                    TRY_CAST(TRY_CAST(v.id_viaje AS DOUBLE) AS BIGINT) AS id_viaje_num,
                    
                    po.latitud AS lat_o, po.longitud AS lon_o, 
                    -- MAGIA ACÁ: Calculamos el H3 de la parada al vuelo igual que en Fig 7 y 8
                    h3_latlng_to_cell_string(TRY_CAST(po.latitud AS DOUBLE), TRY_CAST(po.longitud AS DOUBLE), 8) AS h3_o,
                    
                    pd.latitud AS lat_d, pd.longitud AS lon_d, 
                    h3_latlng_to_cell_string(TRY_CAST(pd.latitud AS DOUBLE), TRY_CAST(pd.longitud AS DOUBLE), 8) AS h3_d
                FROM read_csv_auto('resultados/viajes.csv', ignore_errors=true) v
                LEFT JOIN read_csv_auto('data/paradas.csv', ignore_errors=true) po 
                    ON TRY_CAST(TRY_CAST(v.parada_id_o AS DOUBLE) AS BIGINT) = TRY_CAST(TRY_CAST(po.id AS DOUBLE) AS BIGINT)
                LEFT JOIN read_csv_auto('data/paradas.csv', ignore_errors=true) pd 
                    ON TRY_CAST(TRY_CAST(v.parada_id_d AS DOUBLE) AS BIGINT) = TRY_CAST(TRY_CAST(pd.id AS DOUBLE) AS BIGINT)
            ),
            viajes_filtrados AS (
                SELECT 
                    v.*, 
                    COALESCE(e.hora_viaje, 12) AS hora_viaje 
                FROM viajes_base v
                LEFT JOIN etapas_hora e ON v.id_tarjeta_num = e.id_tarjeta_num AND v.id_viaje_num = e.id_viaje_num
                WHERE v.h3_o = '{hex_objetivo}' OR v.h3_d = '{hex_objetivo}'
            ),
            conexiones AS (
                SELECT 
                    hora_viaje,
                    ROUND(TRY_CAST(lat_d AS DOUBLE), 3) AS lat, 
                    ROUND(TRY_CAST(lon_d AS DOUBLE), 3) AS lon,
                    'Hacia Destino (Saliendo del Hex)' AS direccion
                FROM viajes_filtrados
                WHERE h3_o = '{hex_objetivo}' AND lat_d IS NOT NULL
                
                UNION ALL
                
                SELECT 
                    hora_viaje,
                    ROUND(TRY_CAST(lat_o AS DOUBLE), 3) AS lat, 
                    ROUND(TRY_CAST(lon_o AS DOUBLE), 3) AS lon,
                    'Desde Origen (Llegando al Hex)' AS direccion
                FROM viajes_filtrados
                WHERE h3_d = '{hex_objetivo}' AND lat_o IS NOT NULL
            )
            SELECT 
                hora_viaje, lat, lon, direccion, COUNT(*) AS volumen
            FROM conexiones
            GROUP BY 1, 2, 3, 4
            ORDER BY hora_viaje ASC
        """).df()

        if not od_hex.empty:
            print(f"¡Éxito! Se mapearon {len(od_hex):,} flujos distintos hacia/desde el hexágono.")
            print("Renderizando mapa de flujos (Figura 9)...")
            
            fig9 = px.scatter_map(
                od_hex,
                lat="lat",
                lon="lon",
                color="direccion",
                size="volumen",
                animation_frame="hora_viaje",
                color_discrete_sequence=["#00CC96", "#EF553B"], 
                size_max=30,
                zoom=11,
                center=dict(lat=lat_centro, lon=lon_centro),
                map_style="carto-darkmatter",
                title=f"Figura 9: Orígenes y Destinos por Hora (Hexágono: {hex_objetivo})"
            )
            
            fig9.add_scattermap(
                lat=[lat_centro],
                lon=[lon_centro],
                mode='markers+text',
                marker=dict(size=12, color='white', symbol='star'),
                text=['Hexágono Analizado'],
                textposition='bottom right',
                name='Nodo Objetivo'
            )
            
            fig9.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
            fig9.write_html('resultados/visualizaciones/figura9_od_hexagono.html')
            print("Figura 9 guardada con éxito (figura9_od_hexagono.html).")
            
        else:
            print("Aviso: Aún no se encontraron viajes. Esto significa que las coordenadas de paradas.csv no caen dentro de este hexágono.")
            
    except Exception as e:
        print(f"Error al generar la Figura 9: {e}")

    # 13. FIGURA 10: Mapa Geográfico de Paradas
    # =========================================
    print("\nGenerando Figura 10 (Mapa de todas las paradas)...")
    try:
        # Extraemos las coordenadas directamente del CSV de paradas
        paradas_df = con.execute("""
            SELECT 
                id AS parada_id,
                TRY_CAST(latitud AS DOUBLE) AS lat,
                TRY_CAST(longitud AS DOUBLE) AS lon
            FROM read_csv_auto('data/paradas.csv', ignore_errors=true)
            WHERE latitud IS NOT NULL AND longitud IS NOT NULL
        """).df()

        if not paradas_df.empty:
            print(f"Renderizando mapa con {len(paradas_df):,} paradas (Figura 10)...")
            
            fig10 = px.scatter_map(
                paradas_df,
                lat="lat",
                lon="lon",
                hover_name="parada_id", # Al pasar el mouse, verás el ID de la parada
                color_discrete_sequence=["#00d2ff"], # Un azul/celeste neón para contrastar
                zoom=9,
                center=dict(lat=-34.6037, lon=-58.3816), # Centro en el AMBA
                map_style="carto-darkmatter",
                title="Figura 10: Distribución Geográfica de Todas las Paradas"
            )
            
            # Ajustes finos: hacemos los puntos pequeños (size=3) y con un poco de
            # transparencia (opacity=0.5) para que no saturen la pantalla donde hay mucha densidad
            fig10.update_traces(marker=dict(size=3, opacity=0.5))
            fig10.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
            
            fig10.write_html('resultados/visualizaciones/figura10_mapa_paradas.html')
            print("Figura 10 guardada con éxito (figura10_mapa_paradas.html).")
            
        else:
            print("Aviso: No se encontraron coordenadas válidas en data/paradas.csv.")
            print("Verificá que las columnas se llamen 'latitud' y 'longitud'.")
            
    except Exception as e:
        print(f"Error al generar la Figura 10: {e}")

    # 14. FIGURA 11: Evolución Temporal de un Clúster de H3 (Paradas)
    # ===============================================================
    print("\nGenerando Figura 11 (Animación de Clúster H3 Específico)...")
    try:
        
        # La lista exacta de los hexágonos que querés estudiar
        hex_cluster = [
            '8ac2e3022b17fff', 
            '8ac2e302280ffff', 
            '8ac2e3022b07fff', 
            '8ac2e3022b77fff'
        ]
        
        # Formateamos para SQL: '8ac2...', '8ac2...'
        hex_cluster_str = "','".join(hex_cluster)
        
        print("Agrupando viajes por hora para el clúster (rellenando horas sin actividad)...")
        
        # Magia en SQL: Generamos el cruce de las 24hs con los 4 hexágonos 
        # para que la animación no pierda las formas geométricas a la madrugada.
        cluster_df = con.execute(f"""
            WITH horas AS (
                SELECT unnest(generate_series(0, 23)) AS hora_viaje
            ),
            hexes AS (
                SELECT unnest(LIST_VALUE('{hex_cluster[0]}', '{hex_cluster[1]}', '{hex_cluster[2]}', '{hex_cluster[3]}')) AS h3_index
            ),
            grid_base AS (
                SELECT h.hora_viaje, x.h3_index 
                FROM horas h CROSS JOIN hexes x
            ),
            etapas_hora AS (
                SELECT 
                    TRY_CAST(TRY_CAST(id_tarjeta AS DOUBLE) AS BIGINT) AS id_tarjeta_num,
                    TRY_CAST(TRY_CAST(id_viaje AS DOUBLE) AS BIGINT) AS id_viaje_num,
                    MIN(TRY_CAST(hora AS INTEGER)) AS hora_viaje
                FROM read_csv_auto('resultados/etapas.csv', ignore_errors=true)
                WHERE id_tarjeta IS NOT NULL AND id_viaje IS NOT NULL
                GROUP BY 1, 2
            ),
            viajes_base AS (
                SELECT 
                    TRY_CAST(TRY_CAST(v.id_tarjeta AS DOUBLE) AS BIGINT) AS id_tarjeta_num,
                    TRY_CAST(TRY_CAST(v.id_viaje AS DOUBLE) AS BIGINT) AS id_viaje_num,
                    LOWER(TRIM(po.h3)) AS h3_o,
                    LOWER(TRIM(pd.h3)) AS h3_d
                FROM read_csv_auto('resultados/viajes.csv', ignore_errors=true) v
                LEFT JOIN read_csv_auto('data/paradas.csv', ignore_errors=true) po 
                    ON TRY_CAST(TRY_CAST(v.parada_id_o AS DOUBLE) AS BIGINT) = TRY_CAST(TRY_CAST(po.id AS DOUBLE) AS BIGINT)
                LEFT JOIN read_csv_auto('data/paradas.csv', ignore_errors=true) pd 
                    ON TRY_CAST(TRY_CAST(v.parada_id_d AS DOUBLE) AS BIGINT) = TRY_CAST(TRY_CAST(pd.id AS DOUBLE) AS BIGINT)
            ),
            viajes_filtrados AS (
                SELECT 
                    v.h3_o, v.h3_d, 
                    COALESCE(e.hora_viaje, 12) AS hora_viaje 
                FROM viajes_base v
                LEFT JOIN etapas_hora e ON v.id_tarjeta_num = e.id_tarjeta_num AND v.id_viaje_num = e.id_viaje_num
            ),
            movimientos AS (
                -- Pasajeros que inician su viaje acá
                SELECT hora_viaje, h3_o AS h3_index, COUNT(*) AS vol 
                FROM viajes_filtrados WHERE h3_o IN ('{hex_cluster_str}') GROUP BY 1, 2
                UNION ALL
                -- Pasajeros que finalizan su viaje acá
                SELECT hora_viaje, h3_d AS h3_index, COUNT(*) AS vol 
                FROM viajes_filtrados WHERE h3_d IN ('{hex_cluster_str}') GROUP BY 1, 2
            ),
            movimientos_agrupados AS (
                SELECT hora_viaje, h3_index, SUM(vol) AS total_actividad
                FROM movimientos
                GROUP BY 1, 2
            )
            SELECT 
                g.hora_viaje, 
                g.h3_index, 
                COALESCE(m.total_actividad, 0) AS cantidad_transacciones
            FROM grid_base g
            LEFT JOIN movimientos_agrupados m 
                ON g.hora_viaje = m.hora_viaje AND g.h3_index = m.h3_index
            ORDER BY g.hora_viaje ASC
        """).df()

        if not cluster_df.empty:
            print("Construyendo el GeoJSON para los 4 hexágonos...")
            
            geojson_features = []
            lats, lons = [], []
            
            for hex_id in hex_cluster:
                # Extraemos el punto central para luego poder centrar el mapa exacto en este clúster
                lat, lon = h3.cell_to_latlng(hex_id)
                lats.append(lat)
                lons.append(lon)
                
                # Obtenemos los vértices para dibujar los hexágonos
                boundary_lat_lon = h3.cell_to_boundary(hex_id)
                boundary = [[lon, lat] for lat, lon in boundary_lat_lon]
                boundary.append(boundary[0]) # Cerramos el polígono
                
                geojson_features.append({
                    "type": "Feature",
                    "id": hex_id,
                    "geometry": {
                        "type": "Polygon",
                        "coordinates": [boundary]
                    }
                })
                
            geojson_data = {
                "type": "FeatureCollection",
                "features": geojson_features
            }
            
            # Promediamos las coordenadas para decirle a Plotly dónde enfocar la cámara
            lat_centro = sum(lats) / len(lats)
            lon_centro = sum(lons) / len(lons)

            # Buscamos la hora pico de este grupo para congelar la escala de color
            umbral_maximo = cluster_df['cantidad_transacciones'].max()
            
            print("Renderizando mapa Choropleth Animado (Figura 11)...")
            fig11 = px.choropleth_map(
                cluster_df,
                geojson=geojson_data,
                locations='h3_index',
                color='cantidad_transacciones',
                animation_frame='hora_viaje',
                color_continuous_scale="Inferno", 
                range_color=(0, umbral_maximo if umbral_maximo > 0 else 10), # Mantiene la barra de colores fija
                map_style="carto-darkmatter",
                zoom=15, # Nivel de zoom altísimo porque son pocas manzanas (resolución 10)
                center=dict(lat=lat_centro, lon=lon_centro),
                opacity=0.75,
                title="Figura 11: Actividad por Hora en Clúster Específico de H3"
            )
            
            fig11.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
            fig11.write_html('resultados/visualizaciones/figura11_cluster_h3_animado.html')
            print("Figura 11 guardada con éxito (figura11_cluster_h3_animado.html).")
            
        else:
            print("Aviso: Ocurrió un problema, la base de datos de DuckDB devolvió datos vacíos para el clúster.")
            
    except Exception as e:
        print(f"Error al generar la Figura 11: {e}")

    print("\nEDA finalizado con éxito. Las 9 imágenes fueron guardadas en el directorio actual.")

    # MÉTRICAS TEMPORALES Y DE FRICCIÓN
    # =================================
    print("\nCalculando Métricas de Fricción (Espacial y Temporal)...")
    
    resumen_zona = con.execute("""
        SELECT 
            tipo_viaje,
            MEDIAN((cantidad_etapas / distancia) * 10) AS Mediana_IFI,
            MEDIAN(cantidad_etapas / duracion_horas) AS Mediana_IFT,
            MEDIAN(distancia / duracion_horas) AS Mediana_Velocidad,
            AVG(duracion_horas) * 60 AS Duracion_Promedio_Minutos
        FROM viajes_geo
        WHERE distancia >= 0.5 AND duracion_horas IS NOT NULL
        GROUP BY tipo_viaje
    """).df()

    print("\n--- MÉTRICAS DE FRICCIÓN MEDIANAS POR ZONA GEOGRÁFICA ---")
    print("IFI: Etapas por 10km | IFT: Etapas por Hora | Vel: Km/h en línea recta")
    print(resumen_zona.round(2).to_string(index=False))

    resumen_tarifa = con.execute("""
        SELECT 
            nombre_tarifa,
            COUNT(*) AS Viajes,
            AVG(cantidad_etapas) AS Promedio_Etapas,
            AVG(duracion_horas) * 60 AS Duracion_Media_Minutos,
            MEDIAN((cantidad_etapas / distancia) * 10) AS Mediana_IFI,
            MEDIAN(cantidad_etapas / duracion_horas) AS Mediana_IFT
        FROM viajes_geo
        WHERE distancia >= 0.5 AND duracion_horas IS NOT NULL AND nombre_tarifa != 'Desconocida'
        GROUP BY nombre_tarifa
        ORDER BY Promedio_Etapas DESC
        LIMIT 15
    """).df()

    print("\n--- ANÁLISIS DE PENALIZACIÓN POR TARIFA ESPECÍFICA (Top 15 volúmenes) ---")
    print(resumen_tarifa[['nombre_tarifa', 'Promedio_Etapas', 'Duracion_Media_Minutos', 'Mediana_IFI', 'Mediana_IFT']].round(2).to_string(index=False))

    # DISTANCIAS GLOBALES
    # ===================
    distancias_globales = con.execute("""
        SELECT 
            AVG(distancia) AS media_global,
            MEDIAN(distancia) AS mediana_global
        FROM viajes_geo
        WHERE distancia > 0 AND distancia < 100
    """).fetchone()
    
    print("\n--- DISTANCIA GLOBAL DEL AMBA ---")
    print(f"La distancia MEDIA global es: {distancias_globales[0]:.2f} km")
    print(f"La distancia MEDIANA global es: {distancias_globales[1]:.2f} km")
    print("---------------------------------")

    resumen_distancias = con.execute("""
        SELECT 
            tipo_viaje,
            ROUND(AVG(distancia), 2) AS Media_km,
            ROUND(MEDIAN(distancia), 2) AS Mediana_km,
            COUNT(*) AS Cantidad_Viajes
        FROM viajes_geo
        WHERE distancia > 0 AND distancia < 100
        GROUP BY tipo_viaje
    """).df()

    print("\n--- DISTANCIA POR JURISDICCIÓN ---")
    print(resumen_distancias.to_string(index=False, formatters={'Cantidad_Viajes': '{:,.0f}'.format}))

    # 15. FIGURA 12: Distribución de Fricción Espacial y Temporal (Boxplots)
    # ======================================================================
    print("\nGenerando Figura 12 (Boxplots de IFI e IFT)...")
    try:
        # Obtenemos los datos necesarios
        viajes_friccion = con.execute("""
            SELECT 
                tipo_viaje_detallado,
                (cantidad_etapas / distancia) * 10 AS IFI,
                (cantidad_etapas / duracion_horas) AS IFT
            FROM viajes_geo
            WHERE distancia >= 0.5 AND duracion_horas IS NOT NULL AND tipo_viaje_detallado != 'Desconocido'
        """).df()

        if not viajes_friccion.empty:
            # Mostramos las métricas descriptivas detalladas en la consola
            print("\n--- ESTADÍSTICAS DESCRIPTIVAS DE IFI (Etapas cada 10 km) ---")
            desc_ifi = viajes_friccion.groupby('tipo_viaje_detallado')['IFI'].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
            print(desc_ifi.round(2).to_string())

            print("\n--- ESTADÍSTICAS DESCRIPTIVAS DE IFT (Etapas por hora) ---")
            desc_ift = viajes_friccion.groupby('tipo_viaje_detallado')['IFT'].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
            print(desc_ift.round(2).to_string())

            # Muestreamos para acelerar el ploteado y reducir el tamaño de la figura
            df_sample = viajes_friccion.sample(n=min(100000, len(viajes_friccion)), random_state=42)

            fig12, axes = plt.subplots(1, 2, figsize=(14, 6))

            sns.boxplot(data=df_sample, x='tipo_viaje_detallado', y='IFI', ax=axes[0],
                        order=['Intra-CABA', 'Intra-PBA', 'PBA-CABA', 'CABA-PBA'], hue='tipo_viaje_detallado', legend=False,
                        palette='Set2', showfliers=False)
            axes[0].set_title('Figura 12a: Distribución de IFI (sin outliers)', fontweight='bold')
            axes[0].set_xlabel('Tipo de Viaje')
            axes[0].set_ylabel('IFI (Etapas por 10 km)')

            # Añadir mediana en los boxplots
            medians_ifi = viajes_friccion.groupby('tipo_viaje_detallado')['IFI'].median()
            for i, category in enumerate(['Intra-CABA', 'Intra-PBA', 'PBA-CABA', 'CABA-PBA']):
                val = medians_ifi[category]
                axes[0].text(i, val + 0.1, f'{val:.2f}', ha='center', va='bottom', weight='bold', color='black',
                             bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

            sns.boxplot(data=df_sample, x='tipo_viaje_detallado', y='IFT', ax=axes[1],
                        order=['Intra-CABA', 'Intra-PBA', 'PBA-CABA', 'CABA-PBA'], hue='tipo_viaje_detallado', legend=False,
                        palette='Set2', showfliers=False)
            axes[1].set_title('Figura 12b: Distribución de IFT (sin outliers)', fontweight='bold')
            axes[1].set_xlabel('Tipo de Viaje')
            axes[1].set_ylabel('IFT (Etapas por hora)')

            medians_ift = viajes_friccion.groupby('tipo_viaje_detallado')['IFT'].median()
            for i, category in enumerate(['Intra-CABA', 'Intra-PBA', 'PBA-CABA', 'CABA-PBA']):
                val = medians_ift[category]
                axes[1].text(i, val + 0.05, f'{val:.2f}', ha='center', va='bottom', weight='bold', color='black',
                             bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

            plt.suptitle('Figura 12: Comparación de Índices de Fricción (IFI y IFT)', fontweight='bold')
            plt.tight_layout()
            plt.savefig('resultados/visualizaciones/figura12_boxplot_ifi_ift.png', dpi=300)
            plt.close()
            print("Figura 12 guardada con éxito (figura12_boxplot_ifi_ift.png).")

            # 16. FIGURA 13: Curvas de Densidad de Fricción (KDE)
            # =====================================================
            print("\nGenerando Figura 13 (Curvas de Densidad de IFI e IFT)...")
            fig13, axes13 = plt.subplots(2, 1, figsize=(12, 10))

            # IFI Density
            for cat in ['Intra-CABA', 'Intra-PBA', 'PBA-CABA', 'CABA-PBA']:
                subset = viajes_friccion[viajes_friccion['tipo_viaje_detallado'] == cat]
                sns.kdeplot(data=subset, x='IFI', ax=axes13[0], label=cat, fill=True, alpha=0.2, clip=(0, 10))
            axes13[0].set_title('Figura 13a: Densidad de Probabilidad de IFI (Corte <= 10)', fontweight='bold')
            axes13[0].set_xlabel('IFI (Etapas por 10 km)')
            axes13[0].set_ylabel('Densidad')
            axes13[0].legend(title='Tipo de Viaje')

            # IFT Density
            for cat in ['Intra-CABA', 'Intra-PBA', 'PBA-CABA', 'CABA-PBA']:
                subset = viajes_friccion[viajes_friccion['tipo_viaje_detallado'] == cat]
                sns.kdeplot(data=subset, x='IFT', ax=axes13[1], label=cat, fill=True, alpha=0.2, clip=(0, 6))
            axes13[1].set_title('Figura 13b: Densidad de Probabilidad de IFT (Corte <= 6)', fontweight='bold')
            axes13[1].set_xlabel('IFT (Etapas por hora)')
            axes13[1].set_ylabel('Densidad')
            axes13[1].legend(title='Tipo de Viaje')

            plt.suptitle('Figura 13: Curvas de Densidad de IFI e IFT por Jurisdicción de Viaje', fontweight='bold')
            plt.tight_layout()
            plt.savefig('resultados/visualizaciones/figura13_densidad_ifi_ift.png', dpi=300)
            plt.close()
            print("Figura 13 guardada con éxito (figura13_densidad_ifi_ift.png).")
        else:
            print("Aviso: No se encontraron datos válidos para calcular la fricción.")
    except Exception as e:
        print(f"Error al generar las Figuras de distribución de fricción: {e}")

    # 15. FIGURA 12: Distribución de Fricción Espacial y Temporal (Boxplots)
    # ======================================================================
    print("\nGenerando Figura 12 (Boxplots de IFI e IFT)...")
    try:
        # Obtenemos los datos necesarios
        viajes_friccion = con.execute("""
            SELECT 
                tipo_viaje_detallado,
                (cantidad_etapas / distancia) * 10 AS IFI,
                (cantidad_etapas / duracion_horas) AS IFT
            FROM viajes_geo
            WHERE distancia >= 0.5 AND duracion_horas IS NOT NULL AND tipo_viaje_detallado != 'Desconocido'
        """).df()

        if not viajes_friccion.empty:
            # Mostramos las métricas descriptivas detalladas en la consola
            print("\n--- ESTADÍSTICAS DESCRIPTIVAS DE IFI (Etapas cada 10 km) ---")
            desc_ifi = viajes_friccion.groupby('tipo_viaje_detallado')['IFI'].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
            print(desc_ifi.round(2).to_string())

            print("\n--- ESTADÍSTICAS DESCRIPTIVAS DE IFT (Etapas por hora) ---")
            desc_ift = viajes_friccion.groupby('tipo_viaje_detallado')['IFT'].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
            print(desc_ift.round(2).to_string())

            # Muestreamos para acelerar el ploteado y reducir el tamaño de la figura
            df_sample = viajes_friccion.sample(n=min(100000, len(viajes_friccion)), random_state=42)

            fig12, axes = plt.subplots(1, 2, figsize=(14, 6))

            sns.boxplot(data=df_sample, x='tipo_viaje_detallado', y='IFI', ax=axes[0],
                        order=['Intra-CABA', 'Intra-PBA', 'PBA-CABA', 'CABA-PBA'], hue='tipo_viaje_detallado', legend=False,
                        palette='Set2', showfliers=False)
            axes[0].set_title('Figura 12a: Distribución de IFI (sin outliers)', fontweight='bold')
            axes[0].set_xlabel('Tipo de Viaje')
            axes[0].set_ylabel('IFI (Etapas por 10 km)')

            # Añadir mediana en los boxplots
            medians_ifi = viajes_friccion.groupby('tipo_viaje_detallado')['IFI'].median()
            for i, category in enumerate(['Intra-CABA', 'Intra-PBA', 'PBA-CABA', 'CABA-PBA']):
                val = medians_ifi[category]
                axes[0].text(i, val + 0.1, f'{val:.2f}', ha='center', va='bottom', weight='bold', color='black',
                             bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

            sns.boxplot(data=df_sample, x='tipo_viaje_detallado', y='IFT', ax=axes[1],
                        order=['Intra-CABA', 'Intra-PBA', 'PBA-CABA', 'CABA-PBA'], hue='tipo_viaje_detallado', legend=False,
                        palette='Set2', showfliers=False)
            axes[1].set_title('Figura 12b: Distribución de IFT (sin outliers)', fontweight='bold')
            axes[1].set_xlabel('Tipo de Viaje')
            axes[1].set_ylabel('IFT (Etapas por hora)')

            medians_ift = viajes_friccion.groupby('tipo_viaje_detallado')['IFT'].median()
            for i, category in enumerate(['Intra-CABA', 'Intra-PBA', 'PBA-CABA', 'CABA-PBA']):
                val = medians_ift[category]
                axes[1].text(i, val + 0.05, f'{val:.2f}', ha='center', va='bottom', weight='bold', color='black',
                             bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

            plt.suptitle('Figura 12: Comparación de Índices de Fricción (IFI y IFT)', fontweight='bold')
            plt.tight_layout()
            plt.savefig('resultados/visualizaciones/figura12_boxplot_ifi_ift.png', dpi=300)
            plt.close()
            print("Figura 12 guardada con éxito (figura12_boxplot_ifi_ift.png).")

            # 16. FIGURA 13: Curvas de Densidad de Fricción (KDE)
            # =====================================================
            print("\nGenerando Figura 13 (Curvas de Densidad de IFI e IFT)...")
            fig13, axes13 = plt.subplots(2, 1, figsize=(12, 10))

            # IFI Density
            for cat in ['Intra-CABA', 'Intra-PBA', 'PBA-CABA', 'CABA-PBA']:
                subset = viajes_friccion[viajes_friccion['tipo_viaje_detallado'] == cat]
                sns.kdeplot(data=subset, x='IFI', ax=axes13[0], label=cat, fill=True, alpha=0.2, clip=(0, 10))
            axes13[0].set_title('Figura 13a: Densidad de Probabilidad de IFI (Corte <= 10)', fontweight='bold')
            axes13[0].set_xlabel('IFI (Etapas por 10 km)')
            axes13[0].set_ylabel('Densidad')
            axes13[0].legend(title='Tipo de Viaje')

            # IFT Density
            for cat in ['Intra-CABA', 'Intra-PBA', 'PBA-CABA', 'CABA-PBA']:
                subset = viajes_friccion[viajes_friccion['tipo_viaje_detallado'] == cat]
                sns.kdeplot(data=subset, x='IFT', ax=axes13[1], label=cat, fill=True, alpha=0.2, clip=(0, 6))
            axes13[1].set_title('Figura 13b: Densidad de Probabilidad de IFT (Corte <= 6)', fontweight='bold')
            axes13[1].set_xlabel('IFT (Etapas por hora)')
            axes13[1].set_ylabel('Densidad')
            axes13[1].legend(title='Tipo de Viaje')

            plt.suptitle('Figura 13: Curvas de Densidad de IFI e IFT por Jurisdicción de Viaje', fontweight='bold')
            plt.tight_layout()
            plt.savefig('resultados/visualizaciones/figura13_densidad_ifi_ift.png', dpi=300)
            plt.close()
            print("Figura 13 guardada con éxito (figura13_densidad_ifi_ift.png).")
        else:
            print("Aviso: No se encontraron datos válidos para calcular la fricción.")
    except Exception as e:
        print(f"Error al generar las Figuras de distribución de fricción: {e}")

### ANÁLISIS DE VULNERABILIDAD SOCIO-ESPACIAL Y CENTRALIDAD DE RED (FIGURAS 14, 15 y 16)

Este análisis combina la vulnerabilidad socioeconómica (proporción de viajes con Tarifa Social) y la fricción espacial (IFI) con la teoría de redes y centralidades ponderadas por duración de viaje para mapear zonas postergadas en el AMBA.

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import networkx as nx
import h3
import os

def run_vulnerability_network_analysis():
    print("Iniciando el Análisis de Vulnerabilidad y Centralidad de Redes (Versión Optimizada)...")
    con = duckdb.connect(database=':memory:')

    print("Cargando datasets en DuckDB...")
    try:
        con.execute("CREATE VIEW viajes AS SELECT * FROM read_csv_auto('resultados/viajes.csv', ignore_errors=true)")
        con.execute("CREATE VIEW etapas AS SELECT * FROM read_csv_auto('resultados/etapas.csv', ignore_errors=true)")
        con.execute("CREATE VIEW tarjetas AS SELECT * FROM read_csv_auto('resultados/tarjetas.csv', ignore_errors=true)")
        con.execute("CREATE VIEW indices_h3 AS SELECT * FROM read_csv_auto('data/indices_h3.csv', ignore_errors=true)")
        con.execute("CREATE VIEW paradas AS SELECT * FROM read_csv_auto('data/paradas.csv', ignore_errors=true)")
        con.execute("CREATE VIEW tarifas AS SELECT * FROM read_csv_auto('data/tarifas.csv', ignore_errors=true)")
    except Exception as e:
        print(f"Error al cargar datasets: {e}")
        return

    print("Creando vista viajes_geo...")
    con.execute("""
        CREATE VIEW viajes_geo AS
        WITH viajes_coords AS (
            SELECT 
                v.*,
                po.latitud AS lat_o, po.longitud AS lon_o, po.h3 AS h3_o,
                pd.latitud AS lat_d, pd.longitud AS lon_d, pd.h3 AS h3_d,
                REGEXP_REPLACE(CAST(v.id_tarjeta AS VARCHAR), '\\.0$', '') AS id_tarjeta_str,
                REGEXP_REPLACE(CAST(v.id_viaje AS VARCHAR), '\\.0$', '') AS id_viaje_str,
                6371.0 * 2 * ASIN(SQRT(
                    POWER(SIN(RADIANS(pd.latitud - po.latitud) / 2.0), 2) + 
                    COS(RADIANS(po.latitud)) * COS(RADIANS(pd.latitud)) * POWER(SIN(RADIANS(pd.longitud - po.longitud) / 2.0), 2)
                )) AS distancia,
                CASE WHEN h3_o_idx.provincia ILIKE '%CABA%' OR h3_o_idx.provincia ILIKE '%Ciudad Autónoma%' OR h3_o_idx.provincia ILIKE '%Capital Federal%' THEN 'CABA' ELSE 'PBA' END AS jur_origen,
                CASE WHEN h3_d_idx.provincia ILIKE '%CABA%' OR h3_d_idx.provincia ILIKE '%Ciudad Autónoma%' OR h3_d_idx.provincia ILIKE '%Capital Federal%' THEN 'CABA' ELSE 'PBA' END AS jur_destino
            FROM viajes v
            LEFT JOIN paradas po ON v.parada_id_o = po.id
            LEFT JOIN paradas pd ON v.parada_id_d = pd.id
            LEFT JOIN indices_h3 h3_o_idx ON po.h3 = h3_o_idx.h3
            LEFT JOIN indices_h3 h3_d_idx ON pd.h3 = h3_d_idx.h3
        ),
        viajes_clasificados AS (
            SELECT 
                *,
                CASE 
                    WHEN jur_origen = 'CABA' AND jur_destino = 'CABA' THEN 'Intra-CABA'
                    WHEN jur_origen = 'PBA' AND jur_destino = 'PBA' THEN 'Intra-PBA'
                    WHEN jur_origen IS NOT NULL AND jur_destino IS NOT NULL THEN 'CABA-PBA'
                    ELSE 'Desconocido'
                END AS tipo_viaje,
                CASE 
                    WHEN jur_origen = 'CABA' AND jur_destino = 'CABA' THEN 'Intra-CABA'
                    WHEN jur_origen = 'PBA' AND jur_destino = 'PBA' THEN 'Intra-PBA'
                    WHEN jur_origen = 'PBA' AND jur_destino = 'CABA' THEN 'PBA-CABA'
                    WHEN jur_origen = 'CABA' AND jur_destino = 'PBA' THEN 'CABA-PBA'
                    ELSE 'Desconocido'
                END AS tipo_viaje_detallado,
                CASE 
                    WHEN cantidad_etapas = 1 THEN '1 etapa'
                    WHEN cantidad_etapas = 2 THEN '2 etapas'
                    ELSE '3+ etapas'
                END AS etapas_agrupadas,
                COALESCE(etapas_colectivo, 0) AS c_col,
                COALESCE(etapas_tren, 0) AS c_tren,
                COALESCE(etapas_subte, 0) AS c_sub
            FROM viajes_coords
        ),
        viajes_modos AS (
            SELECT
                *,
                CASE
                    WHEN c_col > 0 AND c_tren = 0 AND c_sub = 0 THEN 'Solo Colectivo'
                    WHEN c_tren > 0 AND c_col = 0 AND c_sub = 0 THEN 'Solo Tren'
                    WHEN c_sub > 0 AND c_col = 0 AND c_tren = 0 THEN 'Solo Subte'
                    WHEN c_col > 0 AND c_tren > 0 AND c_sub = 0 THEN 'Col + Tren'
                    WHEN c_col > 0 AND c_sub > 0 AND c_tren = 0 THEN 'Col + Subte'
                    WHEN c_tren > 0 AND c_sub > 0 AND c_col = 0 THEN 'Tren + Subte'
                    WHEN c_col > 0 AND c_tren > 0 AND c_sub > 0 THEN 'Col + Tren + Subte'
                    ELSE 'Desconocido/Otros'
                END AS modo_transporte
            FROM viajes_clasificados
            WHERE tipo_viaje != 'Desconocido'
        ),
        tarifas_unicas AS (
            SELECT 
                REGEXP_REPLACE(CAST(id_tarjeta AS VARCHAR), '\\.0$', '') AS id_tarjeta_str,
                REGEXP_REPLACE(CAST(id_viaje AS VARCHAR), '\\.0$', '') AS id_viaje_str,
                FIRST(CAST(id_tarifa AS INTEGER)) AS id_tarifa,
                MIN(TRY_CAST(hora AS INTEGER)) AS hora_inicio,
                MAX(TRY_CAST(hora AS INTEGER)) AS hora_fin
            FROM etapas
            WHERE id_tarifa IS NOT NULL
            GROUP BY 1, 2
        )
        SELECT 
            v.*,
            COALESCE(t.nombre, 'Desconocida') AS nombre_tarifa,
            CASE 
                WHEN UPPER(COALESCE(t.nombre, 'Desconocida')) IN ('TARIFA PLENA', 'DESCONOCIDA') THEN 'Tarifa Plena'
                ELSE 'Tarifa Social / Subsidio'
            END AS perfil_tarifario,
            tu.hora_inicio,
            tu.hora_fin,
            CASE 
                WHEN (tu.hora_fin - tu.hora_inicio) < 0 THEN (tu.hora_fin - tu.hora_inicio) + 24 
                WHEN (tu.hora_fin - tu.hora_inicio) = 0 THEN 0.5
                ELSE COALESCE((tu.hora_fin - tu.hora_inicio), 0)
            END AS duracion_horas
        FROM viajes_modos v
        LEFT JOIN tarifas_unicas tu ON v.id_tarjeta_str = tu.id_tarjeta_str AND v.id_viaje_str = tu.id_viaje_str
        LEFT JOIN tarifas t ON tu.id_tarifa = CAST(t.id AS INTEGER)
    """)

    print("[Enfoque 1] Calculando variables de vulnerabilidad por hexágono H3 (Umbral >= 100 viajes)...")
    df_vulnerabilidad = con.execute("""
        SELECT 
            h3_o AS h3_index,
            COUNT(*) AS total_viajes,
            SUM(CASE WHEN perfil_tarifario = 'Tarifa Social / Subsidio' THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS tarifa_social_ratio,
            MEDIAN((cantidad_etapas / distancia) * 10) AS mediana_IFI,
            MEDIAN(cantidad_etapas / duracion_horas) AS mediana_IFT,
            AVG(duracion_horas * 60) AS avg_duration_min,
            AVG(distancia / duracion_horas) AS avg_speed_kmh
        FROM viajes_geo
        WHERE h3_o IS NOT NULL AND distancia >= 0.5 AND duracion_horas > 0
        GROUP BY h3_o
        HAVING COUNT(*) >= 100
    """).df()

    if df_vulnerabilidad.empty:
        print("Error: No se pudieron extraer datos de vulnerabilidad.")
        return

    # Normalizar para armar el Índice de Vulnerabilidad Socio-Espacial (IVSE)
    min_ts, max_ts = df_vulnerabilidad['tarifa_social_ratio'].min(), df_vulnerabilidad['tarifa_social_ratio'].max()
    df_vulnerabilidad['ts_norm'] = (df_vulnerabilidad['tarifa_social_ratio'] - min_ts) / (max_ts - min_ts)
    min_ifi, max_ifi = df_vulnerabilidad['mediana_IFI'].min(), df_vulnerabilidad['mediana_IFI'].max()
    df_vulnerabilidad['ifi_norm'] = (df_vulnerabilidad['mediana_IFI'] - min_ifi) / (max_ifi - min_ifi)
    df_vulnerabilidad['IVSE'] = df_vulnerabilidad['ts_norm'] * df_vulnerabilidad['ifi_norm']

    print("Generando geometries GeoJSON para los hexágonos...")
    geojson_features = []
    for hex_id in df_vulnerabilidad['h3_index'].unique():
        try:
            boundary_lat_lon = h3.cell_to_boundary(hex_id)
            boundary = [[lon, lat] for lat, lon in boundary_lat_lon]
            boundary.append(boundary[0])
            geojson_features.append({
                "type": "Feature",
                "id": hex_id,
                "geometry": {"type": "Polygon", "coordinates": [boundary]}
            })
        except Exception:
            continue

    geojson_data = {"type": "FeatureCollection", "features": geojson_features}

    print("Renderizando Mapa H3 de Vulnerabilidad Socio-Espacial (Figura 14)...")
    umbral_max_ivse = df_vulnerabilidad['IVSE'].quantile(0.98)
    fig14 = px.choropleth_map(
        df_vulnerabilidad, geojson=geojson_data, locations='h3_index', color='IVSE',
        color_continuous_scale="Reds", range_color=(0, umbral_max_ivse),
        map_style="carto-darkmatter", zoom=10, center=dict(lat=-34.6037, lon=-58.3816),
        opacity=0.7, title="Figura 14: Índice de Vulnerabilidad Socio-Espacial (IVSE) por Hexágono H3"
    )
    fig14.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
    fig14.write_html('resultados/visualizaciones/figura14_vulnerabilidad_sql.html')
    print("Figura 14 guardada (resultados/visualizaciones/figura14_vulnerabilidad_sql.html).")

    print("[Enfoque 2] Extrayendo flujo de viajes para la construcción de la red (Filtro enlace >= 30 viajes)...")
    df_edges = con.execute("""
        SELECT h3_o AS source, h3_d AS target, COUNT(*) AS volume, AVG(duracion_horas) AS avg_duration
        FROM viajes_geo
        WHERE h3_o IS NOT NULL AND h3_d IS NOT NULL AND distancia >= 0.5 AND duracion_horas > 0
        GROUP BY h3_o, h3_d
        HAVING COUNT(*) >= 30
    """).df()

    print("Construyendo el Grafo con NetworkX...")
    G = nx.DiGraph()
    for _, row in df_edges.iterrows():
        G.add_edge(row['source'], row['target'], weight=row['avg_duration'], volume=row['volume'])

    print(f"Grafo: {G.number_of_nodes()} nodos y {G.number_of_edges()} enlaces.")
    print("Calculando Cercanía Ponderada...")
    closeness = nx.closeness_centrality(G, distance='weight')
    print("Calculando Intermediación Ponderada...")
    betweenness = nx.betweenness_centrality(G, weight='weight')

    df_centralidades = pd.DataFrame({
        'h3_index': list(G.nodes()),
        'closeness': [closeness.get(node, 0) for node in G.nodes()],
        'betweenness': [betweenness.get(node, 0) for node in G.nodes()]
    })

    df_final = pd.merge(df_vulnerabilidad, df_centralidades, on='h3_index', how='inner')
    df_final.to_csv("resultados/metricas_vulnerabilidad_red.csv", index=False)

    print("Renderizando Mapa H3 de Cercanía Estructural (Figura 15)...")
    umbral_max_close = df_final['closeness'].quantile(0.98)
    fig15 = px.choropleth_map(
        df_final, geojson=geojson_data, locations='h3_index', color='closeness',
        color_continuous_scale="Viridis", range_color=(df_final['closeness'].min(), umbral_max_close),
        map_style="carto-darkmatter", zoom=10, center=dict(lat=-34.6037, lon=-58.3816),
        opacity=0.7, title="Figura 15: Cercanía Estructural de Transporte (Closeness Centrality) - Viridis (Luz = Accesible, Oscuro = Aislado)"
    )
    fig15.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
    fig15.write_html('resultados/visualizaciones/figura15_cercania_red.html')
    print("Figura 15 guardada (resultados/visualizaciones/figura15_cercania_red.html).")

    print("Renderizando Mapa H3 de Intermediación (Figura 16)...")
    umbral_max_between = df_final['betweenness'].quantile(0.98)
    fig16 = px.choropleth_map(
        df_final, geojson=geojson_data, locations='h3_index', color='betweenness',
        color_continuous_scale="Inferno", range_color=(0, umbral_max_between),
        map_style="carto-darkmatter", zoom=10, center=dict(lat=-34.6037, lon=-58.3816),
        opacity=0.7, title="Figura 16: Centralidad de Intermediación (Betweenness) - Inferno (Brillante = Hub, Oscuro = Periférico)"
    )
    fig16.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
    fig16.write_html('resultados/visualizaciones/figura16_intermediacion_red.html')
    print("Figura 16 guardada (resultados/visualizaciones/figura16_intermediacion_red.html).")

    print("\n========================================================")
    print("         RESULTADOS DEL ANÁLISIS DE CORRELACIÓN CRUZADA")
    print("========================================================")
    corr_pears = df_final[['tarifa_social_ratio', 'mediana_IFI', 'closeness', 'betweenness', 'avg_duration_min', 'avg_speed_kmh']].corr(method='pearson')
    corr_spear = df_final[['tarifa_social_ratio', 'mediana_IFI', 'closeness', 'betweenness', 'avg_duration_min', 'avg_speed_kmh']].corr(method='spearman')
    print("\n--- MATRIZ DE CORRELACIÓN DE PEARSON ---")
    print(corr_pears.round(3).to_string())
    print("\n--- MATRIZ DE CORRELACIÓN DE SPEARMAN ---")
    print(corr_spear.round(3).to_string())
    print("========================================================")

if __name__ == "__main__":
    run_vulnerability_network_analysis()

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import networkx as nx
import h3
import os

def run_vulnerability_network_analysis():
    print("Iniciando el Análisis de Vulnerabilidad y Centralidad de Redes (Versión Optimizada)...")

    # Construir el contorno de tierra de AMBA desde indices_h3.csv
    import pandas as pd
    from shapely.geometry import Polygon, Point
    from shapely.ops import unary_union
    from shapely.prepared import prep
    
    df_indices = pd.read_csv("data/indices_h3.csv")
    df_indices['h3_res8'] = df_indices['h3'].apply(lambda x: h3.cell_to_parent(x, 8))
    unique_res8 = df_indices['h3_res8'].unique()
    polygons = []
    for hex_id in unique_res8:
        boundary_lat_lon = h3.cell_to_boundary(hex_id)
        polygons.append(Polygon([(lon, lat) for lat, lon in boundary_lat_lon]))
    land_boundary = unary_union(polygons)
    prepared_land = prep(land_boundary)
    con = duckdb.connect(database=':memory:')

    print("Cargando datasets en DuckDB...")
    try:
        con.execute("CREATE VIEW viajes AS SELECT * FROM read_csv_auto('resultados/viajes.csv', ignore_errors=true)")
        con.execute("CREATE VIEW etapas AS SELECT * FROM read_csv_auto('resultados/etapas.csv', ignore_errors=true)")
        con.execute("CREATE VIEW tarjetas AS SELECT * FROM read_csv_auto('resultados/tarjetas.csv', ignore_errors=true)")
        con.execute("CREATE VIEW indices_h3 AS SELECT * FROM read_csv_auto('data/indices_h3.csv', ignore_errors=true)")
        con.execute("CREATE VIEW paradas AS SELECT * FROM read_csv_auto('data/paradas.csv', ignore_errors=true)")
        con.execute("CREATE VIEW tarifas AS SELECT * FROM read_csv_auto('data/tarifas.csv', ignore_errors=true)")
    except Exception as e:
        print(f"Error al cargar datasets: {e}")
        return

    print("Creando vista viajes_geo...")
    con.execute("""
        CREATE VIEW viajes_geo AS
        WITH viajes_coords AS (
            SELECT 
                v.*,
                po.latitud AS lat_o, po.longitud AS lon_o, po.h3 AS h3_o,
                pd.latitud AS lat_d, pd.longitud AS lon_d, pd.h3 AS h3_d,
                REGEXP_REPLACE(CAST(v.id_tarjeta AS VARCHAR), '\\.0$', '') AS id_tarjeta_str,
                REGEXP_REPLACE(CAST(v.id_viaje AS VARCHAR), '\\.0$', '') AS id_viaje_str,
                6371.0 * 2 * ASIN(SQRT(
                    POWER(SIN(RADIANS(pd.latitud - po.latitud) / 2.0), 2) + 
                    COS(RADIANS(po.latitud)) * COS(RADIANS(pd.latitud)) * POWER(SIN(RADIANS(pd.longitud - po.longitud) / 2.0), 2)
                )) AS distancia,
                CASE WHEN h3_o_idx.provincia ILIKE '%CABA%' OR h3_o_idx.provincia ILIKE '%Ciudad Autónoma%' OR h3_o_idx.provincia ILIKE '%Capital Federal%' THEN 'CABA' ELSE 'PBA' END AS jur_origen,
                CASE WHEN h3_d_idx.provincia ILIKE '%CABA%' OR h3_d_idx.provincia ILIKE '%Ciudad Autónoma%' OR h3_d_idx.provincia ILIKE '%Capital Federal%' THEN 'CABA' ELSE 'PBA' END AS jur_destino
            FROM viajes v
            LEFT JOIN paradas po ON v.parada_id_o = po.id
            LEFT JOIN paradas pd ON v.parada_id_d = pd.id
            LEFT JOIN indices_h3 h3_o_idx ON po.h3 = h3_o_idx.h3
            LEFT JOIN indices_h3 h3_d_idx ON pd.h3 = h3_d_idx.h3
        ),
        viajes_clasificados AS (
            SELECT 
                *,
                CASE 
                    WHEN jur_origen = 'CABA' AND jur_destino = 'CABA' THEN 'Intra-CABA'
                    WHEN jur_origen = 'PBA' AND jur_destino = 'PBA' THEN 'Intra-PBA'
                    WHEN jur_origen IS NOT NULL AND jur_destino IS NOT NULL THEN 'CABA-PBA'
                    ELSE 'Desconocido'
                END AS tipo_viaje,
                CASE 
                    WHEN jur_origen = 'CABA' AND jur_destino = 'CABA' THEN 'Intra-CABA'
                    WHEN jur_origen = 'PBA' AND jur_destino = 'PBA' THEN 'Intra-PBA'
                    WHEN jur_origen = 'PBA' AND jur_destino = 'CABA' THEN 'PBA-CABA'
                    WHEN jur_origen = 'CABA' AND jur_destino = 'PBA' THEN 'CABA-PBA'
                    ELSE 'Desconocido'
                END AS tipo_viaje_detallado,
                CASE 
                    WHEN cantidad_etapas = 1 THEN '1 etapa'
                    WHEN cantidad_etapas = 2 THEN '2 etapas'
                    ELSE '3+ etapas'
                END AS etapas_agrupadas,
                COALESCE(etapas_colectivo, 0) AS c_col,
                COALESCE(etapas_tren, 0) AS c_tren,
                COALESCE(etapas_subte, 0) AS c_sub
            FROM viajes_coords
        ),
        viajes_modos AS (
            SELECT
                *,
                CASE
                    WHEN c_col > 0 AND c_tren = 0 AND c_sub = 0 THEN 'Solo Colectivo'
                    WHEN c_tren > 0 AND c_col = 0 AND c_sub = 0 THEN 'Solo Tren'
                    WHEN c_sub > 0 AND c_col = 0 AND c_tren = 0 THEN 'Solo Subte'
                    WHEN c_col > 0 AND c_tren > 0 AND c_sub = 0 THEN 'Col + Tren'
                    WHEN c_col > 0 AND c_sub > 0 AND c_tren = 0 THEN 'Col + Subte'
                    WHEN c_tren > 0 AND c_sub > 0 AND c_col = 0 THEN 'Tren + Subte'
                    WHEN c_col > 0 AND c_tren > 0 AND c_sub > 0 THEN 'Col + Tren + Subte'
                    ELSE 'Desconocido/Otros'
                END AS modo_transporte
            FROM viajes_clasificados
            WHERE tipo_viaje != 'Desconocido'
        ),
        tarifas_unicas AS (
            SELECT 
                REGEXP_REPLACE(CAST(id_tarjeta AS VARCHAR), '\\.0$', '') AS id_tarjeta_str,
                REGEXP_REPLACE(CAST(id_viaje AS VARCHAR), '\\.0$', '') AS id_viaje_str,
                FIRST(CAST(id_tarifa AS INTEGER)) AS id_tarifa,
                MIN(TRY_CAST(hora AS INTEGER)) AS hora_inicio,
                MAX(TRY_CAST(hora AS INTEGER)) AS hora_fin
            FROM etapas
            WHERE id_tarifa IS NOT NULL
            GROUP BY 1, 2
        )
        SELECT 
            v.*,
            COALESCE(t.nombre, 'Desconocida') AS nombre_tarifa,
            CASE 
                WHEN UPPER(COALESCE(t.nombre, 'Desconocida')) IN ('TARIFA PLENA', 'DESCONOCIDA') THEN 'Tarifa Plena'
                ELSE 'Tarifa Social / Subsidio'
            END AS perfil_tarifario,
            tu.hora_inicio,
            tu.hora_fin,
            CASE 
                WHEN (tu.hora_fin - tu.hora_inicio) < 0 THEN (tu.hora_fin - tu.hora_inicio) + 24 
                WHEN (tu.hora_fin - tu.hora_inicio) = 0 THEN 
                    LEAST(
                        CASE 
                            WHEN modo_transporte = 'Solo Colectivo' THEN (distancia / 15.0) + 0.08
                            WHEN modo_transporte = 'Solo Tren' THEN (distancia / 35.0) + 0.15
                            WHEN modo_transporte = 'Solo Subte' THEN (distancia / 25.0) + 0.08
                            WHEN modo_transporte = 'Col + Tren' THEN (distancia / 25.0) + 0.25
                            WHEN modo_transporte = 'Col + Subte' THEN (distancia / 20.0) + 0.16
                            WHEN modo_transporte = 'Tren + Subte' THEN (distancia / 30.0) + 0.23
                            WHEN modo_transporte = 'Col + Tren + Subte' THEN (distancia / 25.0) + 0.33
                            ELSE (distancia / 18.0) + 0.1
                        END, 
                        0.9
                    )
                ELSE COALESCE((tu.hora_fin - tu.hora_inicio), 0)
            END AS duracion_horas
        FROM viajes_modos v
        LEFT JOIN tarifas_unicas tu ON v.id_tarjeta_str = tu.id_tarjeta_str AND v.id_viaje_str = tu.id_viaje_str
        LEFT JOIN tarifas t ON tu.id_tarifa = CAST(t.id AS INTEGER)
    """)

    print("[Enfoque 1] Calculando variables de vulnerabilidad por hexágono H3 (Umbral >= 100 viajes)...")
    df_vulnerabilidad = con.execute("""
        SELECT 
            h3_o AS h3_index,
            COUNT(*) AS total_viajes,
            SUM(CASE WHEN perfil_tarifario = 'Tarifa Social / Subsidio' THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS tarifa_social_ratio,
            MEDIAN((cantidad_etapas / distancia) * 10) AS mediana_IFI,
            MEDIAN(cantidad_etapas / duracion_horas) AS mediana_IFT,
            AVG(duracion_horas * 60) AS avg_duration_min,
            AVG(distancia / duracion_horas) AS avg_speed_kmh
        FROM viajes_geo
        WHERE h3_o IS NOT NULL AND distancia >= 0.5 AND duracion_horas > 0
        GROUP BY h3_o
        HAVING COUNT(*) >= 100
    """).df()
    if not df_vulnerabilidad.empty:
        # Filtrar celdas fuera del contorno de tierra de AMBA
        df_vulnerabilidad['centroid'] = df_vulnerabilidad['h3_index'].apply(lambda x: h3.cell_to_latlng(x))
        df_vulnerabilidad['in_land'] = df_vulnerabilidad.apply(lambda r: prepared_land.contains(Point(r['centroid'][1], r['centroid'][0])), axis=1)
        df_vulnerabilidad = df_vulnerabilidad[df_vulnerabilidad['in_land']].drop(columns=['centroid', 'in_land'])

    if df_vulnerabilidad.empty:
        print("Error: No se pudieron extraer datos de vulnerabilidad.")
        return

    # Normalizar para armar el Índice de Vulnerabilidad Socio-Espacial (IVSE)
    min_ts, max_ts = df_vulnerabilidad['tarifa_social_ratio'].min(), df_vulnerabilidad['tarifa_social_ratio'].max()
    df_vulnerabilidad['ts_norm'] = (df_vulnerabilidad['tarifa_social_ratio'] - min_ts) / (max_ts - min_ts)
    min_ifi, max_ifi = df_vulnerabilidad['mediana_IFI'].min(), df_vulnerabilidad['mediana_IFI'].max()
    df_vulnerabilidad['ifi_norm'] = (df_vulnerabilidad['mediana_IFI'] - min_ifi) / (max_ifi - min_ifi)
    df_vulnerabilidad['IVSE'] = df_vulnerabilidad['ts_norm'] * df_vulnerabilidad['ifi_norm']

    print("Generando geometries GeoJSON para los hexágonos...")
    geojson_features = []
    for hex_id in df_vulnerabilidad['h3_index'].unique():
        try:
            boundary_lat_lon = h3.cell_to_boundary(hex_id)
            boundary = [[lon, lat] for lat, lon in boundary_lat_lon]
            boundary.append(boundary[0])
            geojson_features.append({
                "type": "Feature",
                "id": hex_id,
                "geometry": {"type": "Polygon", "coordinates": [boundary]}
            })
        except Exception:
            continue

    geojson_data = {"type": "FeatureCollection", "features": geojson_features}

    print("Renderizando Mapa H3 de Vulnerabilidad Socio-Espacial (Figura 14)...")
    umbral_max_ivse = df_vulnerabilidad['IVSE'].quantile(0.98)
    fig14 = px.choropleth_map(
        df_vulnerabilidad, geojson=geojson_data, locations='h3_index', color='IVSE',
        color_continuous_scale="Reds", range_color=(0, umbral_max_ivse),
        map_style="carto-darkmatter", zoom=10, center=dict(lat=-34.6037, lon=-58.3816),
        opacity=0.7, title="Figura 14: Índice de Vulnerabilidad Socio-Espacial (IVSE) por Hexágono H3"
    )
    fig14.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
    fig14.write_html('resultados/visualizaciones/figura14_vulnerabilidad_sql.html')
    print("Figura 14 guardada (figura14_vulnerabilidad_sql.html).")

    print("[Enfoque 2] Extrayendo flujo de viajes para la construcción de la red (Filtro enlace >= 30 viajes)...")
    df_edges = con.execute("""
        SELECT h3_o AS source, h3_d AS target, COUNT(*) AS volume, AVG(duracion_horas) AS avg_duration
        FROM viajes_geo
        WHERE h3_o IS NOT NULL AND h3_d IS NOT NULL AND distancia >= 0.5 AND duracion_horas > 0
        GROUP BY h3_o, h3_d
        HAVING COUNT(*) >= 30
    """).df()

    print("Construyendo el Grafo con NetworkX...")
    G = nx.DiGraph()
    for _, row in df_edges.iterrows():
        G.add_edge(row['source'], row['target'], weight=row['avg_duration'], volume=row['volume'])

    print(f"Grafo: {G.number_of_nodes()} nodos y {G.number_of_edges()} enlaces.")
    print("Calculando Cercanía Ponderada...")
    closeness = nx.closeness_centrality(G, distance='weight')
    print("Calculando Intermediación Ponderada...")
    betweenness = nx.betweenness_centrality(G, weight='weight')

    df_centralidades = pd.DataFrame({
        'h3_index': list(G.nodes()),
        'closeness': [closeness.get(node, 0) for node in G.nodes()],
        'betweenness': [betweenness.get(node, 0) for node in G.nodes()]
    })

    df_final = pd.merge(df_vulnerabilidad, df_centralidades, on='h3_index', how='inner')
    df_final.to_csv("resultados/metricas_vulnerabilidad_red.csv", index=False)

    print("Renderizando Mapa H3 de Cercanía Estructural (Figura 15)...")
    umbral_max_close = df_final['closeness'].quantile(0.98)
    fig15 = px.choropleth_map(
        df_final, geojson=geojson_data, locations='h3_index', color='closeness',
        color_continuous_scale="Viridis", range_color=(df_final['closeness'].min(), umbral_max_close),
        map_style="carto-darkmatter", zoom=10, center=dict(lat=-34.6037, lon=-58.3816),
        opacity=0.7, title="Figura 15: Cercanía Estructural de Transporte (Closeness Centrality) - Viridis (Luz = Accesible, Oscuro = Aislado)"
    )
    fig15.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
    fig15.write_html('resultados/visualizaciones/figura15_cercania_red.html')
    print("Figura 15 guardada (figura15_cercania_red.html).")

    print("Renderizando Mapa H3 de Intermediación (Figura 16)...")
    umbral_max_between = df_final['betweenness'].quantile(0.98)
    fig16 = px.choropleth_map(
        df_final, geojson=geojson_data, locations='h3_index', color='betweenness',
        color_continuous_scale="Inferno", range_color=(0, umbral_max_between),
        map_style="carto-darkmatter", zoom=10, center=dict(lat=-34.6037, lon=-58.3816),
        opacity=0.7, title="Figura 16: Centralidad de Intermediación (Betweenness) - Inferno (Brillante = Hub, Oscuro = Periférico)"
    )
    fig16.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
    fig16.write_html('resultados/visualizaciones/figura16_intermediacion_red.html')
    print("Figura 16 guardada (figura16_intermediacion_red.html).")

    print("\n========================================================")
    print("         RESULTADOS DEL ANÁLISIS DE CORRELACIÓN CRUZADA")
    print("========================================================")
    corr_pears = df_final[['tarifa_social_ratio', 'mediana_IFI', 'closeness', 'betweenness', 'avg_duration_min', 'avg_speed_kmh']].corr(method='pearson')
    corr_spear = df_final[['tarifa_social_ratio', 'mediana_IFI', 'closeness', 'betweenness', 'avg_duration_min', 'avg_speed_kmh']].corr(method='spearman')
    print("\n--- MATRIZ DE CORRELACIÓN DE PEARSON ---")
    print(corr_pears.round(3).to_string())
    print("\n--- MATRIZ DE CORRELACIÓN DE SPEARMAN ---")
    print(corr_spear.round(3).to_string())
    print("========================================================")

if __name__ == "__main__":
    run_vulnerability_network_analysis()